# E4 — Malla 3D → STL imprimible

Toma la malla generada por E3 (`.stl`, `.obj` o `.ply`) y produce un STL watertight listo para imprimir.

```
[malla E3]  →  cargar  →  reparar  →  escalar  →  validar  →  [STL + report.json]
```

**Reparaciones aplicadas:**
1. Hacer manifold (manifold3d — algoritmo ManifoldPlus)
2. Rellenar huecos residuales (trimesh)
3. Fijar normales hacia fuera (trimesh)
4. Eliminar caras degeneradas
5. Conservar componente principal
6. Escalar a tamaño de impresión

**Validación de salida:** watertight · manifold · volumen positivo · euler_number=2

---
## Sección 1 — Setup

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import subprocess

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'numpy']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

# Clonar repo TFM para acceder a los STLs generados por E3
from getpass import getpass
REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR, '-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)

---
## Sección 2 — Configuración

Elige el STL de entrada y el tamaño de impresión.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURACION — solo cambia esta sección
# ══════════════════════════════════════════════════════════════

# Modelo E3 del que vienen los STLs
VERSION_E3 = 'v6_obj_sn'   # o 'v5_obj', etc.

# Ruta al STL de entrada
# Opción A: None → busca todos los STLs generados por E3 para ese VERSION_E3
# Opción B: ruta directa a un .stl/.obj/.ply concreto
RUTA_ENTRADA = None
# RUTA_ENTRADA = '/content/drive/MyDrive/Datos_E2_E3/E3/Raquel/stl/v6_obj_sn/nombre.stl'

# Tamaño de impresión: lado mayor del bounding box en mm (100 = 10 cm)
TAMANO_MM = 100.0

# Rutas de Drive
DRIVE    = '/content/drive/MyDrive'
BASE_E3  = f'{DRIVE}/Datos_E2_E3/E3/Raquel'   # STLs generados por E3
BASE_E4  = f'{DRIVE}/E4/Raquel'                # carpeta E4 de Raquel en Drive

ENTRADA_DIR  = f'{BASE_E3}/stl/{VERSION_E3}'
SALIDA_DIR   = f'E4/stl_reparados/{VERSION_E3}'
SALIDA_DRIVE = f'{BASE_E4}/stl_reparados/{VERSION_E3}'

import os
from pathlib import Path
Path(SALIDA_DIR).mkdir(parents=True, exist_ok=True)

# Listar STLs de entrada disponibles
if RUTA_ENTRADA is None:
    stls_entrada = sorted(Path(ENTRADA_DIR).glob('*.stl')) if Path(ENTRADA_DIR).exists() else []
    if not stls_entrada:
        stls_entrada = sorted(Path(f'E3/stl_generados/{VERSION_E3}').glob('*.stl'))
    print(f'STLs encontrados en {ENTRADA_DIR}:')
    for s in stls_entrada:
        print(f'  {s.name}  ({s.stat().st_size/1024:.0f} KB)')
else:
    stls_entrada = [Path(RUTA_ENTRADA)]
    print(f'Entrada manual: {stls_entrada[0].name}')

print(f'Tamano impresion: {TAMANO_MM} mm')
print(f'Salida local  : {SALIDA_DIR}')
print(f'Salida Drive  : {SALIDA_DRIVE}')

---
## Sección 3 — Cargar y diagnosticar

Inspecciona el estado inicial de cada malla antes de reparar.

In [ ]:
import trimesh
import numpy as np
from pathlib import Path

def diagnosticar(mesh, etiqueta=''):
    wt = mesh.is_watertight
    vol = mesh.volume if wt else None
    eu  = mesh.euler_number
    bb  = mesh.bounding_box.extents
    print(f'  {etiqueta}')
    print(f'    Vertices: {len(mesh.vertices):>6}   Caras: {len(mesh.faces):>6}')
    print(f'    Watertight : {"SI" if wt else "NO"}')
    print(f'    Euler num  : {eu}  (esperado: 2 para impresion)')
    print(f'    Volumen    : {f"{vol*1000:.2f} cm3" if vol else "N/A (no watertight)"}')
    print(f'    BBox       : {bb[0]:.3f} x {bb[1]:.3f} x {bb[2]:.3f} (unidades originales)')
    return {'watertight': wt, 'euler': eu, 'volumen': vol, 'bbox': bb.tolist(),
            'vertices': len(mesh.vertices), 'caras': len(mesh.faces)}

mallas_cargadas = []

for ruta in stls_entrada:
    print(f'\n── {ruta.name} ──')
    try:
        mesh = trimesh.load(str(ruta), force='mesh')
        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(list(mesh.geometry.values()))
        stats_ini = diagnosticar(mesh, 'ANTES de reparar')
        mallas_cargadas.append({'nombre': ruta.stem, 'ruta': str(ruta),
                                 'mesh': mesh, 'stats_ini': stats_ini})
    except Exception as e:
        print(f'  [ERROR] {e}')

print(f'\n{len(mallas_cargadas)} mallas cargadas. Pasa a Seccion 4 para reparar.')

---
## Sección 4 — Reparar

Pipeline de reparación en 4 pasos:

1. **manifold3d** — convierte a manifold (ManifoldPlus): resuelve aristas no-manifold y caras fantasma
2. **fill_holes** — rellena huecos residuales
3. **fix_normals** — orientación coherente hacia fuera
4. **componente principal** — elimina fragmentos sueltos

In [ ]:
import trimesh
import numpy as np
from pathlib import Path

# Intentar manifold3d (ManifoldPlus)
try:
    import manifold3d
    TIENE_MANIFOLD = True
    print('[OK] manifold3d disponible')
except ImportError:
    TIENE_MANIFOLD = False
    print('[WARN] manifold3d no disponible — usando solo trimesh')

def hacer_manifold(mesh):
    if not TIENE_MANIFOLD:
        return mesh, False
    try:
        m = manifold3d.Manifold(
            manifold3d.Mesh(
                vert_properties=np.array(mesh.vertices, dtype=np.float32),
                tri_verts=np.array(mesh.faces, dtype=np.uint32)
            )
        )
        out = m.to_mesh()
        return trimesh.Trimesh(
            vertices=np.array(out.vert_properties),
            faces=np.array(out.tri_verts),
            process=False
        ), True
    except Exception as e:
        print(f'  [WARN manifold3d] {e} — continuando sin manifold')
        return mesh, False

def reparar(mesh):
    reps = []

    # 1. Componente principal (antes de manifold para no procesar basura)
    componentes = mesh.split(only_watertight=False)
    if len(componentes) > 1:
        mesh = max(componentes, key=lambda c: len(c.faces))
        reps.append(f'componente_principal ({len(componentes)} → 1)')

    # 2. ManifoldPlus
    mesh, ok = hacer_manifold(mesh)
    if ok:
        reps.append('manifold3d')

    # 3. Rellenar huecos
    n_antes = len(mesh.faces)
    trimesh.repair.fill_holes(mesh)
    if len(mesh.faces) != n_antes:
        reps.append(f'fill_holes (+{len(mesh.faces)-n_antes} caras)')

    # 4. Fijar normales
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_winding(mesh)
    reps.append('fix_normals')

    # 5. Caras degeneradas
    mask = mesh.nondegenerate_faces()
    n_deg = (~mask).sum()
    if n_deg > 0:
        mesh.update_faces(mask)
        reps.append(f'remove_degenerate ({n_deg} caras)')

    # 6. Procesar (limpia vértices no referenciados, etc.)
    mesh.process(validate=False)

    return mesh, reps

mallas_reparadas = []

for r in mallas_cargadas:
    nombre = r['nombre']
    print(f'\n── {nombre} ──')
    mesh_rep, reps = reparar(r['mesh'])
    print(f'  Reparaciones: {" | ".join(reps)}')

    wt = mesh_rep.is_watertight
    eu = mesh_rep.euler_number
    print(f'  Resultado: watertight={"SI" if wt else "NO"}  euler={eu}  '
          f'vertices={len(mesh_rep.vertices)}  caras={len(mesh_rep.faces)}')

    r['mesh_rep'] = mesh_rep
    r['reparaciones'] = reps
    mallas_reparadas.append(r)

print(f'\n{len(mallas_reparadas)} mallas reparadas.')

---
## Sección 5 — Escalar a tamaño de impresión

El modelo E3 trabaja en unidades de esfera unitaria (radio = 1). Aquí se convierte a mm reales para que la impresora entienda el tamaño.

In [ ]:
import numpy as np

for r in mallas_reparadas:
    mesh = r['mesh_rep']
    nombre = r['nombre']

    # Centrar en origen
    mesh.apply_translation(-mesh.centroid)

    # Escalar: lado mayor del bbox → TAMANO_MM
    lado_max = mesh.bounding_box.extents.max()
    if lado_max > 0:
        factor = TAMANO_MM / lado_max
        mesh.apply_scale(factor)
    else:
        factor = 1.0

    bb = mesh.bounding_box.extents
    r['mesh_final'] = mesh
    r['escala_mm'] = factor
    r['bbox_mm'] = bb.tolist()

    print(f'{nombre}: x{factor:.1f}  →  {bb[0]:.1f} x {bb[1]:.1f} x {bb[2]:.1f} mm')

---
## Sección 6 — Validar y exportar

Comprueba los requisitos de impresión y guarda el STL + `report.json`.

In [ ]:
import json, shutil
from pathlib import Path
from datetime import date

CRITERIOS = {
    'watertight':   'Cerrada sin huecos (obligatorio para impresión)',
    'euler_ok':     'Euler number = 2 (género 0, sólido simple)',
    'volumen_ok':   'Volumen positivo',
    'min_faces':    'Al menos 100 caras',
}

informes = []

for r in mallas_reparadas:
    nombre  = r['nombre']
    mesh    = r['mesh_final']
    print(f'\n── {nombre} ──')

    wt  = mesh.is_watertight
    eu  = mesh.euler_number
    vol = mesh.volume if wt else None
    nf  = len(mesh.faces)

    checks = {
        'watertight': wt,
        'euler_ok':   eu == 2,
        'volumen_ok': vol is not None and vol > 0,
        'min_faces':  nf >= 100,
    }
    apto = all(checks.values())

    for k, v in checks.items():
        print(f'  {"OK" if v else "!!"}  {CRITERIOS[k]}')
    print(f'  --> APTO PARA IMPRIMIR: {"SI" if apto else "NO — requiere revision manual"}')
    if vol:
        print(f'  Volumen: {vol/1000:.2f} cm3  |  BBox: {r["bbox_mm"][0]:.1f}x{r["bbox_mm"][1]:.1f}x{r["bbox_mm"][2]:.1f} mm')

    # Guardar STL
    ruta_stl = Path(SALIDA_DIR) / f'{nombre}_E4.stl'
    mesh.export(str(ruta_stl))
    tam_kb = ruta_stl.stat().st_size / 1024
    print(f'  STL: {ruta_stl.name}  ({tam_kb:.0f} KB)')

    # report.json
    informe = {
        'nombre': nombre,
        'archivo_entrada': r['ruta'],
        'archivo_salida': str(ruta_stl),
        'fecha': str(date.today()),
        'modelo_e3': VERSION_E3,
        'apto_para_imprimir': apto,
        'watertight': wt,
        'euler_number': eu,
        'volumen_cm3': round(vol/1000, 3) if vol else None,
        'n_vertices': len(mesh.vertices),
        'n_caras': nf,
        'bbox_mm': [round(x, 2) for x in r['bbox_mm']],
        'tamano_objetivo_mm': TAMANO_MM,
        'reparaciones': r['reparaciones'],
        'checks': checks,
        'stats_entrada': r['stats_ini'],
    }
    ruta_rep = Path(SALIDA_DIR) / f'{nombre}_report.json'
    with open(ruta_rep, 'w', encoding='utf-8') as f:
        json.dump(informe, f, indent=2, ensure_ascii=False)
    informes.append(informe)

# Copiar a Drive
try:
    Path(SALIDA_DRIVE).mkdir(parents=True, exist_ok=True)
    for f in Path(SALIDA_DIR).glob('*'):
        shutil.copy2(f, Path(SALIDA_DRIVE) / f.name)
    print(f'\nCopiado a Drive: {SALIDA_DRIVE}')
except Exception as e:
    print(f'\n[WARN] Drive: {e}')

# Resumen final
print('\n=== RESUMEN E4 ===')
aptos = sum(1 for i in informes if i['apto_para_imprimir'])
print(f'{aptos}/{len(informes)} mallas aptas para imprimir')
print(f'{"Nombre":<40} {"Apto":>5} {"Water":>6} {"Euler":>6} {"Vol cm3":>8} {"Caras":>7}')
print('-'*72)
for i in informes:
    vol_str = f'{i["volumen_cm3"]:.2f}' if i['volumen_cm3'] else '—'
    print(f'{i["nombre"][:39]:<40} {"SI" if i["apto_para_imprimir"] else "NO":>5} '
          f'{"SI" if i["watertight"] else "NO":>6} {i["euler_number"]:>6} {vol_str:>8} {i["n_caras"]:>7}')

---
## Sección 7 — Visualización 3D final

Renderizado interactivo de las mallas reparadas (rota con el ratón).

In [ ]:
import plotly.graph_objects as go
import numpy as np

_escena = dict(
    xaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    yaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    zaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    bgcolor='#1a1a2e', aspectmode='data'
)

for r, inf in zip(mallas_reparadas, informes):
    mesh   = r['mesh_final']
    nombre = r['nombre']
    verts  = np.array(mesh.vertices)
    faces  = np.array(mesh.faces)

    if len(faces) == 0:
        print(f'[{nombre}] malla vacia'); continue

    # Intensidad por altura (coloring Z)
    intensidad = (verts[:,2] - verts[:,2].min()) / (verts[:,2].ptp() + 1e-8)

    fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#1a6b8a'],[0.5,'#4ecdc4'],[1,'#e8f4f8']],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.8, roughness=0.4, specular=0.5),
        lightposition=dict(x=200, y=300, z=400)
    ))

    bb = inf['bbox_mm']
    apto_str = 'APTO' if inf['apto_para_imprimir'] else 'REVISAR'
    vol_str  = f'{inf["volumen_cm3"]:.2f} cm3' if inf['volumen_cm3'] else 'N/A'

    fig.update_layout(
        scene=_escena,
        title=dict(
            text=f'<b>{nombre}</b>  [{apto_str}]  '
                 f'{bb[0]:.0f}x{bb[1]:.0f}x{bb[2]:.0f} mm  '
                 f'vol={vol_str}  '
                 f'{len(faces)} caras',
            font=dict(color='white', size=11), x=0.5),
        paper_bgcolor='#1a1a2e',
        font=dict(color='white'),
        height=620, width=700,
        margin=dict(l=0, r=0, t=45, b=0)
    )
    fig.show()

---
## Sección 8 — Comparativa antes / después

Muestra lado a lado el diagnóstico de la malla antes y después de reparar.

In [ ]:
import plotly.graph_objects as go

nombres   = [i['nombre'] for i in informes]
wt_antes  = [i['stats_entrada']['watertight'] for i in informes]
wt_despues= [i['watertight'] for i in informes]
eu_antes  = [i['stats_entrada']['euler'] for i in informes]
eu_despues= [i['euler_number'] for i in informes]
vc_antes  = [i['stats_entrada']['vertices'] for i in informes]
vc_despues= [i['n_vertices'] for i in informes]

fig = go.Figure()
fig.add_trace(go.Bar(name='Vértices antes', x=nombres, y=vc_antes,
                     marker_color='#EF9A9A', opacity=0.8))
fig.add_trace(go.Bar(name='Vértices después', x=nombres, y=vc_despues,
                     marker_color='#A5D6A7', opacity=0.8))
fig.update_layout(
    barmode='group',
    title='Vértices antes vs después de reparar',
    xaxis_tickangle=-30,
    height=400
)
fig.show()

print('\nResumen de reparaciones:')
print(f'{"Nombre":<40} {"Water antes":>11} {"Water desp":>10} {"Euler antes":>11} {"Euler desp":>10}')
print('-'*85)
for i in range(len(informes)):
    print(f'{nombres[i][:39]:<40} {"SI" if wt_antes[i] else "NO":>11} '
          f'{"SI" if wt_despues[i] else "NO":>10} '
          f'{eu_antes[i]:>11} {eu_despues[i]:>10}')